# Acomodo mapper

## Librerías

In [2]:
import pandas as pd
import numpy as np
import random

import plotly
import requests
from bs4 import BeautifulSoup
import ruptures as rpt
import yfinance as yf

import plotly.graph_objects as go
import matplotlib.pyplot as plt

import kmapper as km
import subprocess
import sys
import networkx as nx
subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly", "nbformat>=4.2.0"])
import sklearn.cluster

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.cluster import DBSCAN, KMeans
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


## Recolección de datos

### Sacar tickers S&P500

In [3]:

# URL de la lista del S&P 500
url = "https://www.slickcharts.com/sp500"


# Hacer la petición con headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(url, headers=headers)
response.raise_for_status()  # Raise exception for bad status codes

# Parsear el HTML
soup = BeautifulSoup(response.text, 'html.parser')

# Encontrar la tabla
table = soup.find('table')

if table:
    # Leer la tabla con pandas
    df = pd.read_html(str(table))[0]
    
    # Extraer los tickers
    if 'Symbol' in df.columns:
        tickers = df['Symbol'].tolist()
        print(f"Found {len(tickers)} tickers")
        print("First 10 tickers:", tickers[:10])
    else:
        print("Available columns:", df.columns.tolist())

    # Extraer los pesos
    if 'Weight' in df.columns:
        weights = df['Weight'].tolist()
        print(f"Found {len(weights)} weights")
        print("First 10 weights:", weights[:10])
    else:
        print("Available columns:", df.columns.tolist())
else:
    print("No table found on the page")

# En la lista de ticker, reemplazar los "." por "-"
tickers = [ticker.replace('.', '-') for ticker in tickers]
print("Tickers after replacement:", tickers[:10])

Found 503 tickers
First 10 tickers: ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'AVGO', 'GOOGL', 'META', 'GOOG', 'TSLA', 'BRK.B']
Found 503 weights
First 10 weights: ['8.03%', '6.67%', '6.30%', '4.33%', '2.87%', '2.86%', '2.73%', '2.67%', '2.38%', '1.66%']
Tickers after replacement: ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'AVGO', 'GOOGL', 'META', 'GOOG', 'TSLA', 'BRK-B']


### Información financiera

In [4]:
# =======================================================
# 📊 RECOPILACIÓN DE INFORMACIÓN FUNDAMENTAL DE TICKERS
# =======================================================

print("🔍 Recopilando información detallada de todos los tickers del S&P 500...")
print("=" * 80)

ticker_info_db = {}
failed_tickers = []


for idx, ticker in enumerate(tickers, 1):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        # Extraer información relevante
        ticker_info_db[ticker] = {
            # Información básica
            'sector': info.get('sector', 'Unknown'),
            'industry': info.get('industry', 'Unknown'),
            'market_cap': info.get('marketCap', None),
            'country': info.get('country', 'Unknown'),
            'full_name': info.get('longName', ticker),
            
            # Métricas de valoración
            'pe_ratio': info.get('trailingPE', None),
            'forward_pe': info.get('forwardPE', None),
            'peg_ratio': info.get('pegRatio', None),
            'price_to_book': info.get('priceToBook', None),
            'price_to_sales': info.get('priceToSalesTrailing12Months', None),
            'enterprise_value': info.get('enterpriseValue', None),
            'ev_to_ebitda': info.get('enterpriseToEbitda', None),
            
            # Dividendos
            'dividend_yield': info.get('dividendYield', None),
            'dividend_rate': info.get('dividendRate', None),
            'payout_ratio': info.get('payoutRatio', None),
            'five_year_avg_dividend_yield': info.get('fiveYearAvgDividendYield', None),
            
            # Rentabilidad
            'profit_margins': info.get('profitMargins', None),
            'operating_margins': info.get('operatingMargins', None),
            'gross_margins': info.get('grossMargins', None),
            'roe': info.get('returnOnEquity', None),
            'roa': info.get('returnOnAssets', None),
            
            # Crecimiento
            'revenue_growth': info.get('revenueGrowth', None),
            'earnings_growth': info.get('earningsGrowth', None),
            'earnings_quarterly_growth': info.get('earningsQuarterlyGrowth', None),
            
            # Riesgo y volatilidad
            'beta': info.get('beta', None),
            '52week_high': info.get('fiftyTwoWeekHigh', None),
            '52week_low': info.get('fiftyTwoWeekLow', None),
            '52week_change': info.get('52WeekChange', None),
            
            # Liquidez y deuda
            'current_ratio': info.get('currentRatio', None),
            'quick_ratio': info.get('quickRatio', None),
            'debt_to_equity': info.get('debtToEquity', None),
            'total_debt': info.get('totalDebt', None),
            'total_cash': info.get('totalCash', None),
            
            # Analistas y recomendaciones
            'recommendation': info.get('recommendationKey', 'Unknown'),
            'recommendation_mean': info.get('recommendationMean', None),
            'target_mean_price': info.get('targetMeanPrice', None),
            'target_high_price': info.get('targetHighPrice', None),
            'target_low_price': info.get('targetLowPrice', None),
            'number_of_analysts': info.get('numberOfAnalystOpinions', None),
            
            # Información adicional
            'employees': info.get('fullTimeEmployees', None),
            'exchange': info.get('exchange', 'Unknown'),
            'quote_type': info.get('quoteType', 'Unknown'),
        }
        
        
    except Exception as e:
        failed_tickers.append(ticker)
        ticker_info_db[ticker] = {
            'sector': 'Unknown',
            'industry': 'Unknown',
            'error': str(e)
        }

print(f"\n{'='*80}")
print(f"📊 RESUMEN DE RECOPILACIÓN:")
if failed_tickers:
    print(f"   Tickers fallidos: {', '.join(failed_tickers[:10])}{'...' if len(failed_tickers) > 10 else ''}")

# Crear DataFrame para análisis fácil
ticker_info_df = pd.DataFrame.from_dict(ticker_info_db, orient='index')

print(f"\n DataFrame de información creado: {ticker_info_df.shape}")


# Guardar en CSV para uso posterior
ticker_info_df.to_csv('sp500_ticker_info_database.csv')
print("\n Base de datos guardada en 'sp500_ticker_info_database.csv'")

# Mostrar primeras filas
print("\n Primeras filas de la base de datos:")
display(ticker_info_df.head(10))

🔍 Recopilando información detallada de todos los tickers del S&P 500...

📊 RESUMEN DE RECOPILACIÓN:

 DataFrame de información creado: (503, 42)

 Base de datos guardada en 'sp500_ticker_info_database.csv'

 Primeras filas de la base de datos:


,sector,industry,market_cap,country,full_name,pe_ratio,forward_pe,peg_ratio,price_to_book,price_to_sales,...,total_cash,recommendation,recommendation_mean,target_mean_price,target_high_price,target_low_price,number_of_analysts,employees,exchange,quote_type
NVDA,Technology,Semiconductors,4939762892800,United States,NVIDIA Corporation,57.639206,49.245148,None,49.328957,29.898457,...,5.679100e+10,strong_buy,1.34375,225.49614,320.0,100.0,57.0,36000.0,NMS,EQUITY
AAPL,Technology,Consumer Electronics,4027681603584,United States,Apple Inc.,41.246200,32.659443,None,61.250280,9.856669,...,5.537200e+10,buy,2.04167,256.00660,320.0,180.0,41.0,150000.0,NMS,EQUITY
MSFT,Technology,Software - Infrastructure,3908020207616,United States,Microsoft Corporation,37.340910,35.167892,None,11.379102,13.871804,...,9.456500e+10,strong_buy,1.21053,622.81885,730.0,483.0,52.0,228000.0,NMS,EQUITY
AMZN,Consumer Cyclical,Internet Retail,2376782315520,United States,"Amazon.com, Inc.",33.972560,36.237396,None,7.117626,3.547235,...,9.318000e+10,strong_buy,1.28986,267.92062,306.0,230.0,63.0,1546000.0,NMS,EQUITY
AVGO,Technology,Semiconductors,1777828757504,United States,Broadcom Inc.,96.530770,61.016210,None,6.357036,29.667068,...,1.110500e+10,strong_buy,1.28261,392.38165,475.0,273.4,41.0,37000.0,NMS,EQUITY
GOOGL,Communication Services,Internet Content & Information,3407559262208,United States,Alphabet Inc.,27.786774,31.415180,None,10.573211,9.174929,...,9.514800e+10,buy,1.50746,297.28537,350.0,185.0,52.0,187103.0,NMS,EQUITY
META,Communication Services,Internet Content & Information,1674266935296,United States,"Meta Platforms, Inc.",29.450730,26.342688,None,9.246896,9.363700,...,4.707100e+10,strong_buy,1.41176,850.59880,1117.0,616.0,60.0,75945.0,NMS,EQUITY
GOOG,Communication Services,Internet Content & Information,3405202325504,United States,Alphabet Inc.,27.828232,31.497208,None,10.588986,9.168582,...,9.514800e+10,strong_buy,1.50000,292.10000,345.0,185.0,17.0,187103.0,NMS,EQUITY
TSLA,Consumer Cyclical,Auto Manufacturers,1463693082624,United States,"Tesla, Inc.",307.762270,135.833330,None,18.293291,15.305314,...,4.164700e+10,hold,2.61702,391.31805,600.0,120.0,41.0,125665.0,NMS,EQUITY
BRK-B,Financial Services,Insurance - Diversified,1031941586944,United States,Berkshire Hathaway Inc.,16.410150,23.830677,None,0.001031,2.787894,...,3.440910e+11,none,NaN,519.66670,593.0,479.0,3.0,392400.0,NYQ,EQUITY


### Datos históricos separados

In [5]:

START_DATE='2025-04-01'
END_DATE='2025-10-08'
AN_DATE='2025-08-01'

In [6]:
# -----------------------------
# Descargar precios históricos
# -----------------------------
# -----------------------------
# Descargar precios históricos
# -----------------------------
all_tickers_data = yf.download(tickers, start=START_DATE, end=END_DATE, auto_adjust=True)['Close']
all_tickers_data = all_tickers_data.dropna(axis=1)  # eliminar columnas con datos faltantes

# DIVIDIR EN DOS PERIODOS
# Período de análisis: desde START_DATE hasta AN_DATE
analysis_data = all_tickers_data.loc[START_DATE:AN_DATE]
tickers_data = analysis_data  # Datos para entrenamiento/análisis

# Período de validación: desde AN_DATE hasta END_DATE
validation_data = all_tickers_data.loc[AN_DATE:END_DATE]  # Datos para validación

print(f"✅ Datos descargados:")
print(f"   📊 Total: {all_tickers_data.shape} (desde {START_DATE} hasta {END_DATE})")
print(f"   📊 Período de análisis: {tickers_data.shape} (desde {START_DATE} hasta {AN_DATE})")
print(f"   📊 Período de validación: {validation_data.shape} (desde {AN_DATE} hasta {END_DATE})")

[*********************100%***********************]  503 of 503 completed


✅ Datos descargados:
   📊 Total: (131, 503) (desde 2025-04-01 hasta 2025-10-08)
   📊 Período de análisis: (85, 503) (desde 2025-04-01 hasta 2025-08-01)
   📊 Período de validación: (47, 503) (desde 2025-08-01 hasta 2025-10-08)


In [7]:
## Datos S&P500
# Descargar SPY para benchmark
SPY_data = yf.download('SPY', start=START_DATE, end=END_DATE, auto_adjust=True)['Close']

# Verificar si es Serie o DataFrame y convertir apropiadamente
if isinstance(SPY_data, pd.Series):
    SPY = SPY_data.to_frame(name='SPY')
else:
    # Si ya es DataFrame, asegurarse que la columna se llame 'SPY'
    SPY = SPY_data.to_frame() if len(SPY_data.shape) == 1 else SPY_data
    if 'SPY' not in SPY.columns:
        SPY.columns = ['SPY']

# Dividir en períodos
SPY_analysis = SPY.loc[START_DATE:AN_DATE]
SPY_validation = SPY.loc[AN_DATE:END_DATE]

print(f"✅ SPY descargado:")
print(f"   📊 SPY_analysis: {SPY_analysis.shape}")
print(f"   📊 SPY_validation: {SPY_validation.shape}")
print(f"   📋 Columnas: {SPY_analysis.columns.tolist()}")

[*********************100%***********************]  1 of 1 completed

✅ SPY descargado:
   📊 SPY_analysis: (85, 1)
   📊 SPY_validation: (47, 1)
   📋 Columnas: ['SPY']


### Datos de métricas

In [8]:
# =======================================================
# CREAR SUBCARPETA PARA VISUALIZACIONES
# =======================================================

import os

# Crear la subcarpeta para todas las visualizaciones
output_folder = 'visualizaciones_mapper_indice'
os.makedirs(output_folder, exist_ok=True)
print(f"📁 Subcarpeta creada: {output_folder}/")
print(f"🌐 Todas las visualizaciones HTML se guardarán en esta carpeta")

📁 Subcarpeta creada: visualizaciones_mapper_indice/
🌐 Todas las visualizaciones HTML se guardarán en esta carpeta


In [9]:
# Sacamos métricas de cada ticker

financial_metrics = {}
print("\n🔄 Calculando métricas financieras...")
for ticker in tickers_data.columns:
    try:
        prices = tickers_data[ticker].dropna()
        returns = prices.pct_change().dropna()
        
        # Métricas básicas
        total_return = (prices.iloc[-1] / prices.iloc[0]) - 1
        volatility = returns.std() * np.sqrt(252)  # Anualizada
        sharpe = (returns.mean() * 252) / (returns.std() * np.sqrt(252)) if returns.std() > 0 else 0
        
        # Métricas de riesgo
        max_drawdown = ((prices / prices.cummax()) - 1).min()
        var_95 = returns.quantile(0.05)  # Value at Risk 95%
        skewness = returns.skew()
        kurtosis = returns.kurtosis()
        
        # Métricas de tendencia
        returns_positive_ratio = (returns > 0).mean()
        trend_slope = np.polyfit(range(len(prices)), prices.values, 1)[0]
        
        financial_metrics[ticker] = {
            'total_return': total_return,
            'volatility': volatility,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_drawdown,
            'var_95': var_95,
            'skewness': skewness,
            'kurtosis': kurtosis,
            'positive_ratio': returns_positive_ratio,
            'trend_slope': trend_slope / prices.iloc[0]  # Normalizado
        }
    except Exception as e:
        print(f"❌ Error procesando {ticker}: {e}")

# Convertir a DataFrame
metrics_df = pd.DataFrame(financial_metrics).T

# Normalizar datos
scaler = StandardScaler()
metrics_scaled = scaler.fit_transform(metrics_df.fillna(0))

# Preparar datos para KeplerMapper (usar métricas financieras)
datos_metricas = metrics_scaled  # Datos normalizados que ya tenemos
ticker_names = metrics_df.index.tolist()


print(f"📊 Datos preparados: {datos_metricas.shape[0]} tickers, {datos_metricas.shape[1]} métricas")


🔄 Calculando métricas financieras...
📊 Datos preparados: 503 tickers, 9 métricas


## Creación de mappers

### Mappers con datos completos

In [10]:
# Escalamos
data = StandardScaler().fit_transform(tickers_data.T)

# Inicializamos Mapper
mapper = km.KeplerMapper(verbose=1)  # verbose=1 para ver debug info
# Proyección (lens): PCA a 2D
lens = mapper.fit_transform(data, projection=PCA(n_components=2))

# Construcción del grafo con KMeans
graph_hist_1 = mapper.map(
    lens, 
    data, 
    clusterer=KMeans(n_clusters=3, random_state=0),  # ✅ KMeans más confiable
    cover=km.Cover(n_cubes=10, perc_overlap=0.5)
)

print(f"✅ Nodos encontrados: {len(graph_hist_1['nodes'])}")
print(f"✅ Aristas encontradas: {len(graph_hist_1['links'])}")

# Visualización interactiva
mapper.visualize(graph_hist_1, 
                 title="Mapper de series de tiempo",
                 custom_tooltips=np.array(ticker_names),  # Mostrar nombres de tickers
                 path_html="portafolio_v5_visualizaciones/mapper_output.html")

KeplerMapper(verbose=1)
..Composing projection pipeline of length 1:
	Projections: PCA(n_components=2)
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (503, 85)

..Projecting data using: 
	PCA(n_components=2)


..Scaling with: MinMaxScaler()

Mapping on data shaped (503, 85) using lens shaped (503, 2)

Creating 100 hypercubes.

Created 77 edges and 33 nodes in 0:00:01.383001.
✅ Nodos encontrados: 33
✅ Aristas encontradas: 26
Wrote visualization to: portafolio_v5_visualizaciones/mapper_output.html


'<!DOCTYPE html>\n<html>\n\n<head>\n  <meta charset="utf-8">\n  <meta name="generator" content="KeplerMapper">\n  <title>Mapper de series de tiempo | KeplerMapper</title>\n\n  <link rel="icon" type="image/png" href="http://i.imgur.com/axOG6GJ.jpg" />\n\n  <link href=\'https://fonts.googleapis.com/css?family=Roboto+Mono:700,300\' rel=\'stylesheet\' type=\'text/css\'>\n  <style>* {\n  margin: 0;\n  padding: 0;\n}\n\nhtml, body {\n  height: 100%;\n}\n\nbody {\n  font-family: "Roboto Mono", "Helvetica", sans-serif;\n  font-size: 14px;\n}\n\n#logo {\n  width:  85px;\n  height: 85px;\n}\n\n#display {\n  color: #95A5A6;\n  background: #212121;\n}\n\n#header {\n  background: #111111;\n}\n\n#print {\n  color: #000;\n  background: #FFF;\n}\n\nh1 {\n  font-size: 21px;\n  font-weight: 300;\n  font-weight: 300;\n}\n\nh2 {\n  font-size: 18px;\n  padding-bottom: 20px;\n  font-weight: 300;\n}\n\nh3 {\n  font-size: 14px;\n  font-weight: 700;\n  text-transform: uppercase;\n}\n\nh4 {\n  font-size: 13px;\

In [ ]:
# graficamos data
fig = px.scatter_matrix(
    tickers_data.reset_index(),
    dimensions=tickers_data.columns.tolist(),
    title="Matriz de dispersión de precios de tickers",
    labels={col: col for col in tickers_data.columns},
    height=800
)


### Mappers con métricas